### Prepare environment

In [0]:
%run ../environment/prepare_environment

# Multi-Layer Perceptron (MLP) - Foundation & Probability Analysis

## Introduction

Multi-layer perceptron is the simplest form af neural network. This notebook will train an MLP on MNIST and demonstrate confidence-based decision rules.

This notebook will cover:
- Building a multi-layer perceptron neural network from scratch
- Understanding how dense layers, activation functions, and softmax work
- Training with backpropagation and gradient descent
- Interpreting model confidence for business decision-making

## 1. Load and preprocess MNIST data

MNIST is a foundational dataset of 28x28 pixel grayscale images of handwritten digits (0-9) with 60,000 training and 10,000 test images. It is small enough for quick experiments but can demonstrate some neural networks principles.

We will start with loading the dataset and normalizing pixels to [0,1]. Neural networks learn better when inputs are scaled, with large pixel values (0-255) can cause unstable gradients during training. Flattening the 28x28 image to 784 features allows dense layers to process it.

In [0]:
import os
import random
import numpy as np
import tensorflow as tf
import mlflow
import mlflow.tensorflow
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

seed = 15
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
print('train', x_train.shape, 'test', x_test.shape)

## 2. Build MLP model

A multi-layer perceptron (MLP) we will build is a feed-forward neural network composed of:
- **Input layer** (28 x 28 = 784 features): One feature per pixel
- **Flatten layer**: Flattens 2D image to 1D vector
- **Hidden layer** (128 neurons with ReLU): Learns non-linear transformations
- **Output layer** (10 neurons with Softmax): Outputs probability distribution over 10 classes

ReLU (Rectified Linear Unit) introduces non-linearity, allowing the network to learn complex patterns. Without it, the network would be just a linear model. Softmax converts raw outputs (logits) into probabilities that sum to 1, enabling confidence-based decision rules in production.

We will compile the model with the **Adam optimizer**, which adapts learning rates per parameter. The model will learn by minimizing **sparse categorical crossentropy** loss function.

In [0]:
model = keras.Sequential([
    keras.Input(shape=(28, 28)), 
    layers.Flatten(),            
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 3. Model training

During training, the model uses **backpropagation** algorithm to compute gradients of the loss with respect to each weight. The Adam optimizer updates weights in the direction that reduces loss. Each epoch processes the entire training set in batches of 32 images.

Key concepts:
- **Epoch**: One full pass through training data
- **Batch**: Subset of data processed together (32 images)
- **Gradient descent**: Iteratively moves weights to minimize loss
- **Early stopping**: Automatic checks for monitored metric and stop the training if it's not improving
- **Validation split (0.2)**: 20% of training data used to monitor generalization

Watch for these patterns in the output:
- **Normal**: Training loss decreases, validation loss decreases
- **Overfitting**: Training loss continues dropping, but validation loss increases
- **Underfitting**: Both losses remain high

We also plot training history to visualize these curves, making it easy to spot issues and early stopping mechanism which pauses training when validation loss stops improving.

In [0]:
mlflow.tensorflow.autolog(
    silent=True,
    registered_model_name='ai_ml_in_practice.neural_networks_silver.mnist_mlp_model'
    )

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

with mlflow.start_run(run_name='mlp_mnist_extended') as run:
    history = model.fit(
        x_train,
        y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=2
    )
    
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    
    # Log the test metrics
    mlflow.log_metrics({
        "test_accuracy": float(test_acc),
        "test_loss": float(test_loss)
    })
    
    # 1. Accuracy & Loss Summary Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Accuracy subplot
    ax1.plot(history.history['accuracy'], label='Train Acc')
    ax1.plot(history.history['val_accuracy'], label='Val Acc')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    
    # Loss subplot
    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    
    plt.tight_layout()
    
    # Log the figure to MLflow
    mlflow.log_figure(fig, "training_metrics_summary.png")

print(f'Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}')

## 4. Model confidence

The softmax output yields a probability distribution. For an ambiguous digit (e.g., a poorly written 3 that resembles an 8), the model may assign probability to multiple classes.

Why is confidence important?
- **High confidence (>90%)**: Model is certain; could auto-accept in production
- **Low confidence (50-90%)**: Model is uncertain; probably requires human verification
- **Very low confidence**: May indicate input noise or adversarial perturbation

The bar chart shows `P(class|image)` for each digit. If the top probability is only 0.6, the model couldn't confidently classify it. This is where human judgment (e.g., loan officers, doctors) takes over.

An example of this mechanism could be a bank processing checks flags low-confidence predictions for manual review, catching edge cases that purely automated systems would miss.

In [0]:
loaded_model = mlflow.tensorflow.load_model(
    "models:/ai_ml_in_practice.neural_networks_silver.mnist_mlp_model/1"
)

In [0]:
target_digit = 6

digit_indices = np.where(y_test == target_digit)[0]
probs_all_target = loaded_model.predict(x_test[digit_indices], verbose=0)

# Confidence = the highest probability assigned to ANY class
confidences = np.max(probs_all_target, axis=1)

# Sort and pick the index with the LOWEST confidence (most ambiguous)
most_ambiguous_rel_idx = np.argsort(confidences)[0] 
idx = digit_indices[most_ambiguous_rel_idx]
x_amb = x_test[idx:idx+1]
probs = loaded_model.predict(x_amb, verbose=0)[0]

# Visualize digit
plt.figure(figsize=(3,3))
plt.imshow(x_amb.reshape(28,28), cmap='gray')
plt.title(f'Actual: {y_test[idx]} | Pred: {np.argmax(probs)}')
plt.axis('off')
plt.show()

# Visualize probabilities histogram
plt.figure(figsize=(10,4))
bars = plt.bar(range(10), probs, color='tab:blue', alpha=0.7)
bars[y_test[idx]].set_color('tab:green')  # Highlight true label in green
bars[np.argmax(probs)].set_edgecolor('black') # Outline the prediction

plt.xticks(range(10))
plt.ylim(0, 1.1)
plt.title('Model Confidence Distribution')
plt.xlabel('Digit Class')
plt.ylabel('Probability')

for i, v in enumerate(probs):
    plt.text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold' if i == np.argmax(probs) else 'normal')

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()